In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Install required packages
!pip install PyPDF2 nltk sentence-transformers faiss-cpu transformers

# Imports
import os
import json
import PyPDF2
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Download nltk tokenizer data
nltk.download('punkt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 56.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully unins

2025-06-04 12:38:17.159210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749040697.362847      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749040697.417276      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
# ------------------ Load and Extract Text from PDF ------------------
pdf_path = "/kaggle/input/class9-science-combined-pdf/class9_science_combined.pdf"
pdf_reader = PyPDF2.PdfReader(open(pdf_path, "rb"))

print(f"Number of pages in PDF: {len(pdf_reader.pages)}")

all_text = []
for i, page in enumerate(pdf_reader.pages):
    text = page.extract_text()
    if text:
        all_text.append(text)
full_text = "\n".join(all_text)

print(f"Extracted total characters: {len(full_text)}")

# ------------------ Chunking Text ------------------
raw_chunks = full_text.split('\n\n')  # split by paragraphs (double newlines)

final_chunks = []
for raw_chunk in raw_chunks:
    sentences = sent_tokenize(raw_chunk)
    if len(sentences) <= 5:
        final_chunks.append(raw_chunk.strip())
    else:
        for i in range(0, len(sentences), 3):
            chunk = " ".join(sentences[i:i+3]).strip()
            final_chunks.append(chunk)

print(f"Total chunks created: {len(final_chunks)}")


Number of pages in PDF: 152
Extracted total characters: 368796
Total chunks created: 1454


In [4]:
# ------------------ Embedding and Indexing ------------------
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(final_chunks, show_progress_bar=True)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index created and chunks indexed.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

FAISS index created and chunks indexed.


In [5]:
# ------------------ Retrieval Function ------------------
def search_similar_chunks(query, k=5):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(query_embedding, k)
    results = []
    for i in indices[0]:
        results.append({
            "chunk": final_chunks[i],
            "chunk_id": i
        })
    return results

In [6]:
# ------------------ Load LLM Model ------------------
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0


In [7]:
# ------------------ Answer Generation Function ------------------
def generate_answer_with_llm(query, retrieved_chunks, student_level="weak"):
    tone = (
        "Explain clearly and in detail, as if teaching a Class 9 student, using simple language and examples."
    )
    context = "\n".join(chunk["chunk"] for chunk in retrieved_chunks)

    prompt = (
        f"Read the following context carefully and answer the question clearly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        f"{tone}\n"
        f"Answer in detail, using examples and explanations. Write a paragraph or more.\n"
        f"Answer:"
    )

    response = generator(
        prompt,
        max_new_tokens=256,
        do_sample=False,
        num_beams=4,
        early_stopping=True
    )

    answer = response[0]['generated_text']
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()

    return answer


In [10]:
import json
import numpy as np

def log_interaction(user_id, student_level, query, retrieved_chunks, answer, log_file="interaction_log.jsonl"):
    # Convert numpy types to native Python types for JSON serialization
    safe_chunks = []
    for chunk in retrieved_chunks:
        safe_chunk = chunk.copy()
        # Convert chunk_id (likely numpy.int64) to int
        if isinstance(safe_chunk.get("chunk_id"), (np.integer,)):
            safe_chunk["chunk_id"] = int(safe_chunk["chunk_id"])
        safe_chunks.append(safe_chunk)

    log_entry = {
        "user_id": user_id,
        "student_level": student_level,
        "query": query,
        "retrieved_chunks": safe_chunks,
        "answer": answer
    }

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(log_entry) + "\n")


In [11]:
# ------------------ Example Query and Usage ------------------
query = "Why does a ball thrown upwards fall down?"
print(f"\n🔎 Query: {query}")

top_chunks = search_similar_chunks(query, k=5)
print("\n🔹 Retrieved chunks:")
for i, res in enumerate(top_chunks, 1):
    print(f"\nChunk #{i}:\n{res['chunk']}")

student_level = "weak"  # or "strong"
answer = generate_answer_with_llm(query, top_chunks, student_level)

# Log interaction after answer generation
log_interaction(
    user_id="student_1",
    student_level=student_level,
    query=query,
    retrieved_chunks=top_chunks,
    answer=answer
)

print("\n🧠 Chatbot Answer:\n", answer)



🔎 Query: Why does a ball thrown upwards fall down?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔹 Retrieved chunks:

Chunk #1:
·Throw it upwards. ·It reaches a certain height and then it
starts falling down. We have learnt that the earth attracts
objects towards it.

Chunk #2:
SCIENCE 11213. A ball is thrown vertically upwards with a velocity of 49 m/s. Calculate
(i)the maximum height to which it rises,
(ii)the total time it  takes to return to the surface of the
earth.

Chunk #3:
9.1Gravitation
We know that the moon goes ar ound the earth. An object when thrown upwards, reaches a
certain height and then falls downwards. It is
said that when Newton was sitting under a tree,
an apple fell on him.

Chunk #4:
In a high jump athletic
event, the athletes are made to fall either on
a cushioned bed or on a sand bed. This is to
increase the time of the athlete’ s fall to stop
after making the jump. This decreases the rate
of change of momentum and hence the force.

Chunk #5:
I n
doing so, the fielder increases the time during
which the high velocity of the moving ball
decreases to zero.

In [12]:
example_queries = [
    {
        "query": "Why does a ball thrown upwards fall down?",
        "student_level": "weak"
    },
    {
        "query": "Explain the process of photosynthesis in plants.",
        "student_level": "strong"
    },
    {
        "query": "What are Newton's three laws of motion?",
        "student_level": "weak"
    }
]

for i, q in enumerate(example_queries, 1):
    print(f"\n=== Example Query {i} ===")
    print(f"Query: {q['query']}")
    print(f"Student Level: {q['student_level']}")
    
    top_chunks = search_similar_chunks(q['query'], k=5)
    print("\nRetrieved chunks:")
    for idx, chunk in enumerate(top_chunks, 1):
        print(f"Chunk #{idx}:\n{chunk['chunk']}\n")
        
    answer = generate_answer_with_llm(q['query'], top_chunks, q['student_level'])
    log_interaction(
        user_id=f"student_{i}",
        student_level=q['student_level'],
        query=q['query'],
        retrieved_chunks=top_chunks,
        answer=answer
    )
    print("Chatbot Answer:\n", answer)



=== Example Query 1 ===
Query: Why does a ball thrown upwards fall down?
Student Level: weak


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Retrieved chunks:
Chunk #1:
·Throw it upwards. ·It reaches a certain height and then it
starts falling down. We have learnt that the earth attracts
objects towards it.

Chunk #2:
SCIENCE 11213. A ball is thrown vertically upwards with a velocity of 49 m/s. Calculate
(i)the maximum height to which it rises,
(ii)the total time it  takes to return to the surface of the
earth.

Chunk #3:
9.1Gravitation
We know that the moon goes ar ound the earth. An object when thrown upwards, reaches a
certain height and then falls downwards. It is
said that when Newton was sitting under a tree,
an apple fell on him.

Chunk #4:
In a high jump athletic
event, the athletes are made to fall either on
a cushioned bed or on a sand bed. This is to
increase the time of the athlete’ s fall to stop
after making the jump. This decreases the rate
of change of momentum and hence the force.

Chunk #5:
I n
doing so, the fielder increases the time during
which the high velocity of the moving ball
decreases to zero. Th

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Retrieved chunks:
Chunk #1:
SCIENCE 64Recall which gas is required for
photosynthesis. Find out the role of transpiration in plants. Epidermal cells of the roots, whose function
is water absorption, commonly bear long hair -
like parts that greatly increase the total
absorptive surface area.

Chunk #2:
Gr owth of plants and flowering ar e
dependent on sunlight. As we all know, plants
manufactur e their food in sunlight by the
process of photosynthesis. There are some
crops, which are grown in rainy season, called
Q
Reprint 2025-26

Chunk #3:
• A green plant is carrying out photosynthesis. • An engine is pulling a train. Reprint 2025-26

Chunk #4:
THE FUNDAMENT AL UNIT OF LIFE 59• Chromoplasts that contain chlorophyll are called chloroplasts
and they perform photosynthesis. • The primary function of leucoplasts is storage. • Most mature plant cells have a large central vacuole that
helps to maintain the turgidity of the cell and stores
important substances including wastes.

Chunk #5:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Retrieved chunks:
Chunk #1:
These thr ee laws ar e known as
Newton’s laws of motion. The first law of
motion is stated as:
An object r emains in a state of r est or of
unifor m motion in a straight line unless
compelled to change that state by an
applied force. In other words, all objects resist a change
in their state of motion .

Chunk #2:
FORCE AND LAWS OF MOTION 93 9393 9393motion gives us a method to measure the force
acting on an object as a product of its mass
and acceleration. The second law of motion is often seen in
action in our everyday life. Have you noticed
that while catching a fast moving cricket ball,
a fielder in the ground gradually pulls his
hands backwards with the moving ball?

Chunk #3:
SCIENCE 100We have lear nt about the motion of objects and
force as the cause of motion. W e have lear nt
that a force is needed to change the speed or
the dir ection of motion of an object. W e always
observe that an object dropped from a height
falls towar ds the earth.

Chunk 